In [14]:
# Use pre-trained model

import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tqdm import tqdm

BATCH_SIZE = 16
MAX_LENGTH = 256
TEST_DATASET_PATH = "D:/comp-6713-industry-project/processed_data/processed_work_arrangements_test_set.json"

class JobDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def load_json_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    texts = []
    labels = []
    for item in data:
        clean_text = item['text'].replace('\r', ' ').replace('\n', ' ')
        texts.append(clean_text)
        labels.append(item['status'])

    return texts, labels

def generate_test_report():
    test_texts, test_labels = load_json_data(TEST_DATASET_PATH)
    label_encoder = LabelEncoder()
    test_encoded = label_encoder.fit_transform(test_labels)

    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=len(label_encoder.classes_)
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print("Using device:", device)

    test_dataset = JobDataset(test_texts, test_encoded, tokenizer, MAX_LENGTH)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Generating Test Report"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)

    print("\nClassification Report:")
    print(classification_report(
        test_encoded,
        all_preds,
        target_names=label_encoder.classes_
    ))

if __name__ == "__main__":
    generate_test_report()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


Generating Test Report: 100%|████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  6.94it/s]


Classification Report:
              precision    recall  f1-score   support

      Hybrid       0.39      0.70      0.50        27
      OnSite       0.33      0.17      0.23        46
      Remote       0.15      0.15      0.15        26

    accuracy                           0.31        99
   macro avg       0.29      0.34      0.29        99
weighted avg       0.30      0.31      0.28        99



In [ ]:
# Use fine-turned model

import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tqdm import tqdm

BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 2e-5
MAX_LENGTH = 256
RANDOM_SEED = 17

TRAIN_DATASET_PATH = "D:/comp-6713-industry-project/processed_data/processed_work_arrangements_development_set.json"
TEST_DATASET_PATH = "D:/comp-6713-industry-project/processed_data/processed_work_arrangements_test_set.json"

class JobDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def load_json_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    texts = []
    labels = []
    for item in data:
        clean_text = item['text'].replace('\r', ' ').replace('\n', ' ')
        texts.append(clean_text)
        labels.append(item['status'])

    return texts, labels

label_encoder = LabelEncoder()

train_texts, train_labels = load_json_data(TRAIN_DATASET_PATH)
label_encoder.fit(train_labels)
encoded_train_labels = label_encoder.transform(train_labels)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(label_encoder.classes_))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using device:", device)

train_dataset = JobDataset(train_texts, encoded_train_labels, tokenizer, MAX_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    progress_bar = tqdm(train_loader, desc=f'Train Epoch {epoch + 1}/{EPOCHS}')

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        optimizer.step()

        progress_bar.set_postfix({'train_loss': f'{loss.item():.3f}'})

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Epoch {epoch + 1} | Train Loss: {avg_train_loss:.4f}")

FINAL_MODEL_PATH = "./final_model"
model.save_pretrained(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)

def generate_test_report(model_path=FINAL_MODEL_PATH):
    test_texts, test_labels = load_json_data(TEST_DATASET_PATH)
    test_encoded = label_encoder.transform(test_labels)
    
    test_dataset = JobDataset(test_texts, test_encoded, tokenizer, MAX_LENGTH)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    model = BertForSequenceClassification.from_pretrained(model_path).to(device)

    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Generating Test Report"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
    
    print("\nClassification Report:")
    print(classification_report(
        test_encoded,
        all_preds,
        target_names=label_encoder.classes_
    ))

if __name__ == "__main__":
    generate_test_report()
